# 08 | Publicação e geração operacional

POC semissintética • contrato provisório • sem homologação acadêmica.
Execute em kernel reiniciado, de cima para baixo. As credenciais vêm do ambiente.


**Objetivo:** disponibilizar versões aprovadas e gerar a proposta pela API. **Entradas:** campanha e decisões07. Por padrão consulta apenas; altere os controles explicitamente. **Esperado:** versão publicada distinta de residente.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.application_client import call, context, save_context

from src.workbench.academic_ui import campaign_result
result=campaign_result()
DEPLOY_DEMO=os.getenv('V4_DEPLOY_DEMO','false').lower()=='true'
GENERATE=os.getenv('V4_GENERATE','false').lower()=='true'
GENERATION_KEY=os.getenv('V4_GENERATION_KEY','v4-notebook-proposal-001')
selectors=[{k:m[k] for k in ('model_name','model_version')} for m in result['models']]
if DEPLOY_DEMO:
    for index,selector in enumerate(selectors):
        call('POST',f"/v1/serving/models/{selector['model_name']}/deploy",dict(**selector,policy='PRELOAD' if index==0 else 'LAZY'),
             key=f"v4-notebook-deploy-{selector['model_name']}-{selector['model_version']}",operator=True)
display(pd.DataFrame(call('GET','/v1/serving/models')))
display(call('GET','/v1/serving/runtime'))


,model_key,model_version,course_code,validation_status,run_id,snapshot_id,approval_status,deployment_status,policy,expires_at
0,hi.demo.v4.c2248.components,1,2248,VALIDATED,8f70f100788c4731ae6cde89ae8fab1b,9158d4db-c80b-5638-8aa0-9467c60f6107,None,None,None,None
1,hi.demo.v4.c2248.components,2,2248,VALIDATED,c7fdde7a3ab547a3a52e417d491b7005,9158d4db-c80b-5638-8aa0-9467c60f6107,None,DEPLOYED,PRELOAD,2026-12-05T05:55:51.204861+00:00
2,hi.demo.v4.c2248.components,3,2248,VALIDATED,ddee196b563d436b81a24f5fb5b29026,9158d4db-c80b-5638-8aa0-9467c60f6107,None,None,None,None
3,hi.demo.v4.c2248.components,4,2248,VALIDATED,9772e1d9aa3d4c7eae0d1b94645dc420,9158d4db-c80b-5638-8aa0-9467c60f6107,None,DEPLOYED,PRELOAD,2026-12-05T06:07:12.909551+00:00
4,hi.demo.v4.c2248.components,5,2248,VALIDATED,000ce5f9d9b5456588d88722bab64d9d,9158d4db-c80b-5638-8aa0-9467c60f6107,None,None,None,None
5,hi.demo.v4.c2248.components,6,2248,VALIDATED,ebae875becef44089adf28bc37d0955d,9158d4db-c80b-5638-8aa0-9467c60f6107,None,DEPLOYED,PRELOAD,2026-12-05T06:21:57.061225+00:00
6,hi.demo.v4.c2248.components,7,2248,VALIDATED,ada578740d04465198c1a749d35fd913,9158d4db-c80b-5638-8aa0-9467c60f6107,None,None,None,None
7,hi.demo.v4.c2248.components,8,2248,VALIDATED,9112032e221e47c08e0a6bf64a37a6ff,9158d4db-c80b-5638-8aa0-9467c60f6107,APPROVED,DEPLOYED,PRELOAD,2026-12-05T06:29:08.222895+00:00
8,hi.demo.v4.c2287.components,1,2287,VALIDATED,d623100c3246445aa389cb825cc367ff,9158d4db-c80b-5638-8aa0-9467c60f6107,None,None,None,None
9,hi.demo.v4.c2287.components,2,2287,VALIDATED,12d0c41d37b545abb98e54c19f3cf3ec,9158d4db-c80b-5638-8aa0-9467c60f6107,None,DEPLOYED,LAZY,2026-12-05T05:55:52.486520+00:00


{'versions': [{'model_name': 'hi.demo.v4.c2248.components',
   'model_version': '6',
   'loaded_at_utc': '2026-11-05T06:27:29.526377+00:00',
   'last_used_at': '2026-11-05T06:27:29.526387+00:00',
   'load_seconds': 0.10769196299952455,
   'hits': 0,
   'misses': 1}],
 'cache': {'hits': 0, 'misses': 1, 'errors': 0, 'currsize': 1, 'maxsize': 4}}

**Executar:** geração HTTP, aguardando job persistido. **Resultado:**70 ofertas/107 origens, `proposal_id` salvo automaticamente para09. Mesma chave recupera; nova chave gera nova proposta, sem sobrescrever anteriores.

In [2]:

import time
if GENERATE:
    payload=dict(schema_version='poc-planilha-turmas-v2',snapshot_id=result['snapshot_id'],cycle_code='2024-2',proposal_revision=1,
                 model_selectors=selectors,requested_by='jupyter-poc')
    job=call('POST','/v2/proposals/generations',payload,key=GENERATION_KEY)
    for _ in range(90):
        job=call('GET',job['status_url'])
        if job['status'] in ('COMPLETED','FAILED'):
            break
        time.sleep(2)
    if job['status']!='COMPLETED':
        raise RuntimeError('Geração não concluída: '+str({k:job[k] for k in ('job_id','status','error_code')}))
    save_context(proposal_id=job['proposal_id'])
    display({k:job[k] for k in ('job_id','proposal_id','status','proposal_url')})
else:
    display('READ_ONLY: geração não solicitada. Use os controles após aprovação.')


{'job_id': '41f15eac-1dc7-44e0-b360-db0e50da8406',
 'proposal_id': '41f15eac-1dc7-44e0-b360-db0e50da8406',
 'status': 'COMPLETED',
 'proposal_url': '/v2/proposals/41f15eac-1dc7-44e0-b360-db0e50da8406/application-view'}

**Onde conferir:** [MLflow](http://localhost:5000), [RustFS](http://localhost:9001), [API](http://localhost:8000/docs), [Logs](http://localhost:8088). Em falha: `docker compose logs --tail 100 api inference worker mlflow`.